# DUT RevB (with PMU) input-referred noise capture

This notebook acquires **CH0** from **PXI1Slot2** at 50 kSa/s for 22 s (1,100,000 samples), then stores the voltage waveform as `float64` in TDMS.

Before running the acquisition cell, close InstrumentStudio so it does not hold the NI-SCOPE session. The DUT is assumed to use a closed-loop gain of 1001 with a 15 MHz op-amp GBW (estimated closed-loop bandwidth: about 15 kHz). Existing TDMS files are never overwritten.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
import subprocess
import sys

import numpy as np
import niscope
from nptdms import ChannelObject, GroupObject, RootObject, TdmsFile, TdmsWriter

if version("nptdms") != "1.11.0":
    raise RuntimeError(
        f"This notebook requires nptdms==1.11.0; found {version('nptdms')}. "
        f"Install it with: {sys.executable} -m pip install nptdms==1.11.0"
    )

project_candidates = [Path.cwd(), Path.cwd() / "Unicorn_CSA_RevB-COB"]
PROJECT_DIR = next(
    (path.resolve() for path in project_candidates if (path / "analyze_dut_revb_pmu_noise.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Run this notebook from the project or Unicorn_CSA_RevB-COB directory.")

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"niscope: {version('niscope')}")
print(f"nptdms: {version('nptdms')}")
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
RESOURCE_NAME = "PXI1Slot2"
CHANNEL_NAME = "0"
TDMS_GROUP_NAME = "DUT RevB with PMU"
TDMS_CHANNEL_NAME = "CH0"

SAMPLE_RATE_HZ = 50_000.0
DURATION_S = 22.0
NUM_SAMPLES = 1_100_000
VERTICAL_RANGE_VPP = 2.0
VERTICAL_OFFSET_V = 0.0
INPUT_IMPEDANCE_OHM = 1_000_000.0
CLOSED_LOOP_GAIN = 1001.0
OP_AMP_GBW_HZ = 15_000_000.0
ESTIMATED_CLOSED_LOOP_BANDWIDTH_HZ = OP_AMP_GBW_HZ / CLOSED_LOOP_GAIN
FETCH_TIMEOUT_S = 40.0

assert int(SAMPLE_RATE_HZ * DURATION_S) == NUM_SAMPLES
print(f"Requested acquisition: {NUM_SAMPLES:,} points at {SAMPLE_RATE_HZ:,.0f} Sa/s ({DURATION_S:g} s)")
print(f"Estimated closed-loop bandwidth: {ESTIMATED_CLOSED_LOOP_BANDWIDTH_HZ:,.3f} Hz")

## Acquire and write TDMS

The acquisition is held in a preallocated NumPy `float64` array and filled with `fetch_into()`. The file is first written as `*.partial.tdms`, reopened and checked, and only then renamed to its final timestamped name.

In [ ]:
tasklist = subprocess.run(
    ["tasklist", "/FI", "IMAGENAME eq InstrumentStudio.exe"],
    capture_output=True,
    text=True,
    check=False,
)
if "instrumentstudio.exe" in tasklist.stdout.lower():
    raise RuntimeError("Close InstrumentStudio before acquiring from PXI1Slot2.")

samples_v = np.empty(NUM_SAMPLES, dtype=np.float64)
acquisition_start_utc = datetime.now(timezone.utc)

with niscope.Session(RESOURCE_NAME, reset_device=False) as scope:
    channel = scope.channels[CHANNEL_NAME]
    channel.configure_vertical(
        range=VERTICAL_RANGE_VPP,
        coupling=niscope.VerticalCoupling.DC,
        offset=VERTICAL_OFFSET_V,
        probe_attenuation=1.0,
        enabled=True,
    )
    channel.configure_chan_characteristics(
        input_impedance=INPUT_IMPEDANCE_OHM,
        max_input_frequency=-1.0,
    )
    scope.configure_horizontal_timing(
        min_sample_rate=SAMPLE_RATE_HZ,
        min_num_pts=NUM_SAMPLES,
        ref_position=0.0,
        num_records=1,
        enforce_realtime=True,
    )
    scope.configure_trigger_immediate()

    actual_sample_rate_hz = float(scope.horz_sample_rate)
    actual_record_length = int(scope.horz_record_length)
    actual_vertical_range_vpp = float(channel.vertical_range)
    actual_input_impedance_ohm = float(channel.input_impedance)

    instrument_metadata = {
        "instrument_manufacturer": str(scope.instrument_manufacturer),
        "instrument_model": str(scope.instrument_model),
        "instrument_serial_number": str(scope.serial_number),
        "instrument_firmware_revision": str(scope.instrument_firmware_revision),
        "ni_scope_driver_version": str(scope.specific_driver_revision),
    }

    print(f"Instrument: {instrument_metadata['instrument_manufacturer']} {instrument_metadata['instrument_model']}")
    print(f"Serial number: {instrument_metadata['instrument_serial_number']}")
    print(f"NI-SCOPE driver: {instrument_metadata['ni_scope_driver_version']}")
    print(f"Actual sample rate: {actual_sample_rate_hz:,.9f} Sa/s")
    print(f"Actual record length: {actual_record_length:,}")
    print(f"Actual vertical range: {actual_vertical_range_vpp:g} Vpp")
    print(f"Actual input impedance: {actual_input_impedance_ohm:g} ohm")

    if not np.isclose(actual_sample_rate_hz, SAMPLE_RATE_HZ, rtol=1e-9, atol=1e-6):
        raise RuntimeError(
            f"Actual sample rate is {actual_sample_rate_hz} Sa/s, not {SAMPLE_RATE_HZ} Sa/s; no TDMS written."
        )
    if actual_record_length < NUM_SAMPLES:
        raise RuntimeError(
            f"Actual record length {actual_record_length:,} is below {NUM_SAMPLES:,}; no TDMS written."
        )
    if not np.isclose(actual_vertical_range_vpp, VERTICAL_RANGE_VPP):
        raise RuntimeError(
            f"Actual vertical range is {actual_vertical_range_vpp} Vpp, not {VERTICAL_RANGE_VPP} Vpp; no TDMS written."
        )
    if not np.isclose(actual_input_impedance_ohm, INPUT_IMPEDANCE_OHM):
        raise RuntimeError(
            f"Actual input impedance is {actual_input_impedance_ohm} ohm, not {INPUT_IMPEDANCE_OHM} ohm; no TDMS written."
        )

    print("Acquiring 22 seconds of CH0 data...")
    with scope.initiate():
        waveform_info = channel.fetch_into(samples_v, timeout=FETCH_TIMEOUT_S)[0]

if samples_v.dtype != np.dtype(np.float64) or samples_v.size != NUM_SAMPLES:
    raise RuntimeError(f"Unexpected acquired array: shape={samples_v.shape}, dtype={samples_v.dtype}")
if not np.all(np.isfinite(samples_v)):
    raise RuntimeError("Acquired waveform contains NaN or infinite values; no TDMS written.")

sample_interval_s = float(waveform_info.x_increment)
if not np.isclose(sample_interval_s, 1.0 / actual_sample_rate_hz, rtol=1e-9, atol=1e-15):
    raise RuntimeError("Waveform time interval does not match the reported sample rate; no TDMS written.")

timestamp = acquisition_start_utc.strftime("%Y%m%dT%H%M%SZ")
tdms_path = DATA_DIR / f"DUT_RevB_with_PMU_CH0_{timestamp}.tdms"
partial_tdms_path = DATA_DIR / f"DUT_RevB_with_PMU_CH0_{timestamp}.partial.tdms"
if tdms_path.exists() or partial_tdms_path.exists():
    raise FileExistsError(f"Refusing to overwrite an existing acquisition for {timestamp}.")

root_properties = {
    "test_target": "DUT RevB (with PMU)",
    "dut_revision": "RevB",
    "pmu_enabled": True,
    "closed_loop_gain_v_per_v": CLOSED_LOOP_GAIN,
    "op_amp_gbw_hz": OP_AMP_GBW_HZ,
    "estimated_closed_loop_bandwidth_hz": ESTIMATED_CLOSED_LOOP_BANDWIDTH_HZ,
    "resource_name": RESOURCE_NAME,
    "acquisition_start_utc": acquisition_start_utc.replace(tzinfo=None),
    **instrument_metadata,
}
group_properties = {
    "description": "DUT RevB with PMU, op-amp closed-loop gain 1001 noise capture",
}
channel_properties = {
    "unit_string": "V",
    "wf_xname": "Time",
    "wf_xunit_string": "s",
    "wf_increment": sample_interval_s,
    "wf_start_offset": float(waveform_info.relative_initial_x),
    "wf_start_time": acquisition_start_utc.replace(tzinfo=None),
    "wf_samples": NUM_SAMPLES,
    "requested_sample_rate_hz": SAMPLE_RATE_HZ,
    "actual_sample_rate_hz": actual_sample_rate_hz,
    "requested_duration_s": DURATION_S,
    "actual_duration_s": NUM_SAMPLES / actual_sample_rate_hz,
    "actual_record_length": actual_record_length,
    "vertical_range_vpp": actual_vertical_range_vpp,
    "vertical_offset_v": VERTICAL_OFFSET_V,
    "input_impedance_ohm": actual_input_impedance_ohm,
    "coupling": "DC",
    "max_input_frequency_hz": -1.0,
    "trigger_type": "Immediate",
    "physical_channel": CHANNEL_NAME,
    "fetch_dtype": str(samples_v.dtype),
}

with TdmsWriter(partial_tdms_path, index_file=False) as writer:
    writer.write_segment(
        [
            RootObject(properties=root_properties),
            GroupObject(TDMS_GROUP_NAME, properties=group_properties),
            ChannelObject(
                TDMS_GROUP_NAME,
                TDMS_CHANNEL_NAME,
                samples_v,
                properties=channel_properties,
            ),
        ]
    )

verification_tdms = TdmsFile.read(partial_tdms_path)
verification_channel = verification_tdms[TDMS_GROUP_NAME][TDMS_CHANNEL_NAME]
stored_samples_v = verification_channel[:]
stored_interval_s = float(verification_channel.properties["wf_increment"])

if stored_samples_v.dtype != np.dtype(np.float64):
    raise RuntimeError(f"TDMS dtype verification failed: {stored_samples_v.dtype}")
if stored_samples_v.size != NUM_SAMPLES:
    raise RuntimeError(f"TDMS sample-count verification failed: {stored_samples_v.size:,}")
if not np.isclose(stored_interval_s, sample_interval_s, rtol=0.0, atol=1e-15):
    raise RuntimeError(f"TDMS sample-interval verification failed: {stored_interval_s}")
if not np.array_equal(stored_samples_v, samples_v):
    raise RuntimeError("TDMS round-trip comparison failed; the partial file was not promoted.")

partial_tdms_path.rename(tdms_path)
print(f"Verified TDMS written: {tdms_path}")
print(f"File size: {tdms_path.stat().st_size / 1e6:.3f} MB")

In [ ]:
dc_mean_v = float(np.mean(stored_samples_v))
ac_samples_v = stored_samples_v - dc_mean_v
ac_rms_v = float(np.sqrt(np.mean(ac_samples_v**2)))
peak_to_peak_v = float(np.ptp(stored_samples_v))
peak_from_center_v = float(np.max(np.abs(stored_samples_v - VERTICAL_OFFSET_V)))
near_clipping = peak_from_center_v >= 0.98 * (actual_vertical_range_vpp / 2.0)

print(f"Verified dtype: {stored_samples_v.dtype}")
print(f"Verified samples: {stored_samples_v.size:,}")
print(f"Verified interval: {stored_interval_s:.12g} s ({1.0 / stored_interval_s:,.9f} Sa/s)")
print(f"DC mean: {dc_mean_v:.12g} V")
print(f"AC RMS: {ac_rms_v:.12g} V")
print(f"Peak-to-peak: {peak_to_peak_v:.12g} Vpp")
print(f"Within 2% of vertical full scale: {near_clipping}")
if near_clipping:
    print("WARNING: The capture is close to clipping; check the DUT offset and vertical range.")

## Run the Welch PSD analysis

The standalone script reads the sample interval and closed-loop gain from TDMS metadata, calculates one-sided Welch PSD, converts it to input-referred ASD in nV/√Hz, and saves the log-log plot. It does not remove narrowband peaks or deconvolve the estimated 15 kHz bandwidth.

In [ ]:
analysis_script = PROJECT_DIR / "analyze_dut_revb_pmu_noise.py"
command = [
    sys.executable,
    str(analysis_script),
    str(tdms_path),
    "--output-dir",
    str(RESULTS_DIR),
]
analysis = subprocess.run(command, capture_output=True, text=True, check=True)
print(analysis.stdout)
if analysis.stderr:
    print(analysis.stderr, file=sys.stderr)

plot_path = RESULTS_DIR / f"{tdms_path.stem}_input_noise_asd.png"
from IPython.display import Image, display
display(Image(filename=str(plot_path)))